## Find notable weather weeks

In [1]:
import pandas as pd 
import sqlite3

In [3]:
# Connect to your database
db_path = "load_cases.db"
conn = sqlite3.connect(db_path)

# 1️⃣ List all tables in the database
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in the database:")
print(tables)

# 2️⃣ Preview the first few rows of one table (e.g., weather_0)
df_test = pd.read_sql("SELECT * FROM weather_0 LIMIT 5;", conn)
print("\nFirst 5 rows of weather_0:")
print(df_test)

# Close connection when done
conn.close()


Tables in the database:
          name
0   weather_m6
1   weather_m5
2   weather_m4
3   weather_m3
4   weather_m2
5   weather_m1
6    weather_0
7   weather_p1
8   weather_p2
9   weather_p3
10  weather_p4
11  weather_p5
12  weather_p6

First 5 rows of weather_0:
   season                       date        thi          load
0  Summer  1993-06-01 00:00:00+00:00  70.389621  95954.689559
1  Summer  1993-06-01 01:00:00+00:00  69.858066  95301.481007
2  Summer  1993-06-01 02:00:00+00:00  69.498836  95114.095027
3  Summer  1993-06-01 03:00:00+00:00  69.192772  94900.807633
4  Summer  1993-06-01 04:00:00+00:00  69.112543  94531.000259


In [4]:

# Define your tables
tables = [
    "weather_m6", "weather_m5", "weather_m4", "weather_m3", "weather_m2", "weather_m1",
    "weather_0",
    "weather_p1", "weather_p2", "weather_p3", "weather_p4", "weather_p5", "weather_p6"
]

def get_worst_loads(cursor, season, limit=30):
    # Construct a UNION ALL query across all 13 tables
    # Adding 'table_source' so you know which weather realization the peak came from
    query_parts = []
    for table in tables:
        query_parts.append(f"SELECT '{table}' as src, date, thi, load FROM {table} WHERE season = ?")
    
    full_query = " UNION ALL ".join(query_parts)
    final_query = f"SELECT * FROM ({full_query}) ORDER BY load DESC LIMIT {limit}"
    
    cursor.execute(final_query, ([season] * len(tables)))
    return cursor.fetchall()

# Connect and Execute
try:
    conn = sqlite3.connect('load_cases.db')
    cursor = conn.cursor()

    print("--- 30 WORST SUMMER LOADS ---")
    summer_peaks = get_worst_loads(cursor, "Summer")
    for row in summer_peaks:
        print(f"Source: {row[0]} | Date: {row[1]} | THI: {row[2]} | Load: {row[3]}")

    print("\n--- 30 WORST WINTER LOADS ---")
    winter_peaks = get_worst_loads(cursor, "Winter")
    for row in winter_peaks:
        print(f"Source: {row[0]} | Date: {row[1]} | THI: {row[2]} | Load: {row[3]}")

except sqlite3.Error as e:
    print(f"Database error: {e}")
finally:
    if conn:
        conn.close()

--- 30 WORST SUMMER LOADS ---


Source: weather_m1 | Date: 2025-07-24 17:00:00+00:00 | THI: 85.22472322249149 | Load: 153848.8773100579
Source: weather_p6 | Date: 2025-07-31 17:00:00+00:00 | THI: 85.22472322249149 | Load: 153848.8773100579
Source: weather_m4 | Date: 2025-07-21 17:00:00+00:00 | THI: 85.22472322249149 | Load: 153836.08305521295
Source: weather_p3 | Date: 2025-07-28 17:00:00+00:00 | THI: 85.22472322249149 | Load: 153836.08305521295
Source: weather_m6 | Date: 2010-07-29 18:00:00+00:00 | THI: 85.777221536556 | Load: 153795.31958725848
Source: weather_m6 | Date: 2010-07-29 17:00:00+00:00 | THI: 85.504356707632 | Load: 153755.19371578595
Source: weather_m6 | Date: 2011-07-14 16:00:00+00:00 | THI: 85.291000528 | Load: 153393.44424405097
Source: weather_m1 | Date: 1999-07-29 17:00:00+00:00 | THI: 85.3602338548355 | Load: 153393.44424405097
Source: weather_p1 | Date: 2011-07-21 16:00:00+00:00 | THI: 85.291000528 | Load: 153393.44424405097
Source: weather_0 | Date: 2025-07-25 17:00:00+00:00 | THI: 85.2247232224

In [18]:
# Worst post-solar loads
tables = [
    "weather_m6", "weather_m5", "weather_m4", "weather_m3", "weather_m2", "weather_m1",
    "weather_0",
    "weather_p1", "weather_p2", "weather_p3", "weather_p4", "weather_p5", "weather_p6"
]

def get_worst_offpeak_summer(cursor, limit=30):
    query_parts = []
    
    # SQLite strftime('%H', date) returns the hour as a string '00' through '23'
    # We filter out 10-18 inclusive
    for table in tables:
        query_parts.append(f"""
            SELECT '{table}' as src, date, thi, load 
            FROM {table} 
            WHERE season = 'Winter' 
            AND (strftime('%H', date) < '10' OR strftime('%H', date) > '18')
        """)
    
    full_union = " UNION ALL ".join(query_parts)
    final_query = f"SELECT * FROM ({full_union}) ORDER BY load DESC LIMIT {limit}"
    
    cursor.execute(final_query)
    return cursor.fetchall()

try:
    conn = sqlite3.connect('load_cases.db')
    cursor = conn.cursor()

    print("--- 30 WORST SUMMER LOADS (Hours 0-9 & 19-23) ---")
    results = get_worst_offpeak_summer(cursor)
    
    # Header for clarity
    print(f"{'Source':<15} | {'Date/Time':<25} | {'THI':<6} | {'Load':<10}")
    print("-" * 65)
    
    for row in results:
        print(f"{row[0]:<15} | {row[1]:<25} | {row[2]:<6} | {row[3]:<10}")

except sqlite3.Error as e:
    print(f"Database error: {e}")
finally:
    if conn:
        conn.close()

--- 30 WORST SUMMER LOADS (Hours 0-9 & 19-23) ---
Source          | Date/Time                 | THI    | Load      
-----------------------------------------------------------------
weather_m1      | 2015-02-19 06:00:00+00:00 | 9.6133995 | 135830.6539697522
weather_m1      | 2015-02-19 07:00:00+00:00 | 9.253399 | 135830.6539697522
weather_p6      | 2015-02-26 06:00:00+00:00 | 9.6133995 | 135830.6539697522
weather_p6      | 2015-02-26 07:00:00+00:00 | 9.253399 | 135830.6539697522
weather_m1      | 2015-02-19 08:00:00+00:00 | 8.443399 | 135658.99566249253
weather_p6      | 2015-02-26 08:00:00+00:00 | 8.443399 | 135658.99566249253
weather_m1      | 2015-02-19 09:00:00+00:00 | 7.5434 | 135331.37983979785
weather_p6      | 2015-02-26 09:00:00+00:00 | 7.5434 | 135331.37983979785
weather_m6      | 2014-01-16 20:00:00+00:00 | 22.9226 | 135108.20505817948
weather_m6      | 2014-01-16 21:00:00+00:00 | 23.0126 | 135108.20505817948
weather_p1      | 2014-01-23 20:00:00+00:00 | 22.9226 | 135108.205

In [19]:
# Worst post-battery loads 
tables = [
    "weather_m6", "weather_m5", "weather_m4", "weather_m3", "weather_m2", "weather_m1",
    "weather_0",
    "weather_p1", "weather_p2", "weather_p3", "weather_p4", "weather_p5", "weather_p6"
]

def get_worst_offpeak_summer(cursor, limit=30):
    query_parts = []
    
    # SQLite strftime('%H', date) returns the hour as a string '00' through '23'
    # We filter out 10-18 inclusive
    for table in tables:
        query_parts.append(f"""
            SELECT '{table}' as src, date, thi, load 
            FROM {table} 
            WHERE season = 'Summer' 
            AND (strftime('%H', date) < '10' OR strftime('%H', date) > '22')
        """)
    
    full_union = " UNION ALL ".join(query_parts)
    final_query = f"SELECT * FROM ({full_union}) ORDER BY load DESC LIMIT {limit}"
    
    cursor.execute(final_query)
    return cursor.fetchall()

try:
    conn = sqlite3.connect('load_cases.db')
    cursor = conn.cursor()

    print("--- 30 WORST SUMMER LOADS (Hours 0-9 & 19-23) ---")
    results = get_worst_offpeak_summer(cursor)
    
    # Header for clarity
    print(f"{'Source':<15} | {'Date/Time':<25} | {'THI':<6} | {'Load':<10}")
    print("-" * 65)
    
    for row in results:
        print(f"{row[0]:<15} | {row[1]:<25} | {row[2]:<6} | {row[3]:<10}")

except sqlite3.Error as e:
    print(f"Database error: {e}")
finally:
    if conn:
        conn.close()

--- 30 WORST SUMMER LOADS (Hours 0-9 & 19-23) ---
Source          | Date/Time                 | THI    | Load      
-----------------------------------------------------------------
weather_0       | 1999-07-29 23:00:00+00:00 | 85.18062242778751 | 135805.89093498516
weather_m3      | 1999-07-26 23:00:00+00:00 | 85.18062242778751 | 135356.6107949663
weather_m4      | 2014-07-22 23:00:00+00:00 | 82.13828079127677 | 135356.48701254316
weather_p3      | 2014-07-29 23:00:00+00:00 | 82.13828079127677 | 135356.48701254316
weather_m6      | 1999-07-23 23:00:00+00:00 | 85.18062242778751 | 135342.63794440063
weather_p1      | 1999-07-30 23:00:00+00:00 | 85.18062242778751 | 135342.63794440063
weather_m3      | 2014-07-23 23:00:00+00:00 | 82.13828079127677 | 135302.15706962618
weather_p4      | 2014-07-30 23:00:00+00:00 | 82.13828079127677 | 135302.15706962618
weather_m5      | 2011-07-28 23:00:00+00:00 | 84.51901424198375 | 134952.7415163579
weather_m2      | 2014-07-24 23:00:00+00:00 | 82.138280

In [8]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

def get_load_week(table_name, target_date_str, db_path='load_cases.db'):
    """
    Retrieves a week of hourly data (3 days before to 3 days after) 
    from a specific table in the load_cases database.
    """
    # 1. Parse the input date
    target_date = datetime.strptime(target_date_str, "%Y-%m-%d")
    
    # 2. Calculate the date range (3 days before and 3 days after)
    start_date = (target_date - timedelta(days=3)).strftime("%Y-%m-%d 00:00:00")
    end_date = (target_date + timedelta(days=3)).strftime("%Y-%m-%d 23:59:59")
    
    # 3. Connect to the database
    conn = sqlite3.connect(db_path)
    
    # 4. Construct the query
    # Using parameterized queries to prevent SQL injection
    query = f"""
    SELECT season, date, thi, load 
    FROM {table_name} 
    WHERE date BETWEEN ? AND ?
    ORDER BY date ASC
    """
    
    try:
        # 5. Load data into DataFrame
        df = pd.read_sql_query(query, conn, params=(start_date, end_date))
        return df
    finally:
        conn.close()



In [20]:
# Summer:
jul_2025 = get_load_week("weather_m1", "2025-07-24")
jul_2010 = get_load_week("weather_m6", "2010-07-29")
jul_2011 = get_load_week("weather_m6", "2011-07-14")
jul_1999 = get_load_week("weather_m1", "1999-07-29")
jul_2011_2 = get_load_week("weather_0", "2011-07-07")
jul_2005 = get_load_week("weather_m3", "2005-07-21")
jul_2012 = get_load_week("weather_p6", "2012-07-05")
jul_2014 = get_load_week("weather_m4", "2014-07-22")
jul_2011_3 = get_load_week("weather_m5", "2011-07-28")
jul_2018 = get_load_week("weather_p6", "2018-07-05")

summer_weeks = [
    jul_2025, jul_2010, jul_2011, jul_1999, jul_2011_2, jul_2005,
    jul_2012, jul_2014, jul_2011_3, jul_2018
]

In [21]:
# Winter: 
jan_2019 = get_load_week("weather_m4", "2019-01-17") 
jan_2019_2 = get_load_week("weather_p3", "2019-01-24") 
feb_1996 = get_load_week("weather_m4", "1996-02-01") 
feb_2015 = get_load_week("weather_m1", "2015-02-19")
jan_2014 = get_load_week("weather_m6", "2014-01-16")
jan_1994 = get_load_week("weather_0", "1994-01-20")
jan_2018 = get_load_week("weather_m2", "2018-01-04") 
feb_1996_2 = get_load_week("weather_p3", "1996-02-08")
feb_2015_2 = get_load_week("weather_p6", "2015-02-26")
jan_2014_2 = get_load_week("weather_p1", "2014-01-23")

winter_weeks = [
    jan_2019, jan_2019_2, feb_1996, feb_2015, jan_2014, jan_1994, jan_2018,
    feb_1996_2, feb_2015_2, jan_2014_2
]

In [22]:
import sqlite3
import pandas as pd

# 1. Create a mapping of the DataFrames to their primary base names
# This allows us to handle the duplicate month naming logic
summer_mapping = {
    "July 2025": jul_2025, "July 2010": jul_2010, "July 2011": [jul_2011, jul_2011_2, jul_2011_3],
    "July 1999": jul_1999, "July 2005": jul_2005, "July 2012": jul_2012,
    "July 2014": jul_2014, "July 2018": jul_2018
}

winter_mapping = {
    "January 2019": [jan_2019, jan_2019_2], "February 1996": [feb_1996, feb_1996_2],
    "February 2015": [feb_2015, feb_2015_2], "January 2014": [jan_2014, jan_2014_2],
    "January 1994": jan_1994, "January 2018": jan_2018
}

def clean_and_save(mapping, db_name="notable_weeks.db"):
    conn = sqlite3.connect(db_name)
    
    for base_name, dfs in mapping.items():
        # Ensure we are working with a list even if there is only one DF
        if not isinstance(dfs, list):
            dfs = [dfs]
        
        for i, df in enumerate(dfs):
            # Drop the 'season' column
            if 'season' in df.columns:
                df = df.drop(columns=['season'])
            
            # Logic for suffix: only add A, B, C if more than one DF exists for that month
            if len(dfs) > 1:
                alphabet = "ABCDE"
                table_name = f"{base_name} {alphabet[i]}"
            else:
                table_name = base_name
            
            # Save to SQL
            df.to_sql(table_name, conn, if_exists='replace', index=False)
            print(f"Saved table: {table_name}")

    conn.close()

# Execute the save
clean_and_save({**summer_mapping, **winter_mapping})

Saved table: July 2025
Saved table: July 2010
Saved table: July 2011 A
Saved table: July 2011 B
Saved table: July 2011 C
Saved table: July 1999
Saved table: July 2005
Saved table: July 2012
Saved table: July 2014
Saved table: July 2018
Saved table: January 2019 A
Saved table: January 2019 B
Saved table: February 1996 A
Saved table: February 1996 B
Saved table: February 2015 A
Saved table: February 2015 B
Saved table: January 2014 A
Saved table: January 2014 B
Saved table: January 1994
Saved table: January 2018


In [23]:
import sqlite3
import pandas as pd

def get_all_tables_as_dfs(db_path):
    # 1. Connect to the database
    conn = sqlite3.connect(db_path)
    
    # 2. Query the names of all tables in the database
    # The 'sqlite_master' table holds the database schema
    query = "SELECT name FROM sqlite_master WHERE type='table';"
    table_names = pd.read_sql_query(query, conn)['name'].tolist()
    
    # 3. Iterate through names and load each into a DataFrame
    list_of_dfs = []
    for table in table_names:
        df = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)
        list_of_dfs.append(df)
        print(f"Loaded table: {table}")

    # 4. Close the connection
    conn.close()
    
    return list_of_dfs

# Usage
path = 'notable_weeks.db'
dfs = get_all_tables_as_dfs(path)

Loaded table: July 2025
Loaded table: July 2010
Loaded table: July 2011 A
Loaded table: July 2011 B
Loaded table: July 2011 C
Loaded table: July 1999
Loaded table: July 2005
Loaded table: July 2012
Loaded table: July 2014
Loaded table: July 2018
Loaded table: January 2019 A
Loaded table: January 2019 B
Loaded table: February 1996 A
Loaded table: February 1996 B
Loaded table: February 2015 A
Loaded table: February 2015 B
Loaded table: January 2014 A
Loaded table: January 2014 B
Loaded table: January 1994
Loaded table: January 2018


In [26]:
percentages = pd.read_csv("zonal_percentages.csv", header=None).T

new_headers = [] 
for i in range(1, 23): 
    new_headers.append(f"Demand_MW_z{i}")

percentages.columns = new_headers

print(percentages.tail())

    Demand_MW_z1  Demand_MW_z2  Demand_MW_z3  Demand_MW_z4  Demand_MW_z5  \
19      0.015447      0.120049      0.063589      0.006947      0.008089   
20      0.015210      0.123861      0.064953      0.007094      0.007974   
21      0.015013      0.126374      0.066125      0.007221      0.007891   
22      0.014833      0.126697      0.066679      0.007281      0.007832   
23      0.014719      0.125312      0.066496      0.007262      0.007799   

    Demand_MW_z6  Demand_MW_z7  Demand_MW_z8  Demand_MW_z9  Demand_MW_z10  \
19      0.083817      0.011448      0.102701      0.063483       0.081212   
20      0.082636      0.011852      0.101408      0.063852       0.081677   
21      0.081778      0.012152      0.100238      0.064163       0.082070   
22      0.081165      0.012345      0.099220      0.064605       0.082635   
23      0.080831      0.012427      0.098444      0.065073       0.083237   

    ...  Demand_MW_z13  Demand_MW_z14  Demand_MW_z15  Demand_MW_z16  \
19  ...  

In [27]:
import pandas as pd
import numpy as np

# Assume list_of_dfs contains your 20 DataFrames (168 rows each)
# Assume zones_df is your 24x27 percentage profile

# 1. Tile the 24-hour zone percentages to create a 168-hour profile
# np.tile repeats the rows 7 times
tiled_zones = np.tile(percentages.values, (7, 1))

# 2. Convert tiled numpy array back to a DataFrame for easier indexing
tiled_zones_df = pd.DataFrame(tiled_zones, columns=percentages.columns)

output_dfs = []

# 3. Process each of the 20 DataFrames
for df in dfs:
    # Extract the 'load' column as a series
    total_load = df['load']
    
    # 4. Multiply total_load (168, 1) by tiled_zones_df (168, 27)
    # Using .multiply with axis=0 aligns the 168 rows correctly
    zone_load_df = tiled_zones_df.multiply(total_load, axis=0)
    
    output_dfs.append(zone_load_df)

# Example: Check the first zone of the first processed dataframe
print(output_dfs[0].head())

   Demand_MW_z1  Demand_MW_z2  Demand_MW_z3  Demand_MW_z4  Demand_MW_z5  \
0   1390.165430  11763.924991   6269.970674    684.881983    738.151986   
1   1376.626324  11513.293489   6176.305212    674.731702    732.922929   
2   1319.947327  10858.516473   5864.340765    640.736350    706.276785   
3   1295.805992  10348.411312   5639.493347    616.297271    700.888246   
4   1370.173908  10511.553218   5792.572731    633.176781    748.127933   

   Demand_MW_z6  Demand_MW_z7  Demand_MW_z8  Demand_MW_z9  Demand_MW_z10  ...  \
0   7656.079138   1178.718274   9295.552224   6221.508731    7958.531453  ...   
1   7606.125588   1160.440983   9191.117982   6206.898896    7940.155072  ...   
2   7333.476620   1095.886974   8800.502363   5976.902850    7646.232285  ...   
3   7281.888361   1041.660227   8628.887117   5865.209096    7503.827286  ...   
4   7776.655008   1048.847744   9103.312403   6159.337268    7880.838003  ...   

   Demand_MW_z13  Demand_MW_z14  Demand_MW_z15  Demand_MW_z16 

In [28]:
import os

# Create a folder to store the results if it doesn't exist
output_folder = "load_zone_results"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Assuming output_dfs is the list of 20 processed DataFrames
for i, df in enumerate(output_dfs):
    # Indices 0-9 are Summer
    if i < 10:
        season = "Summer"
        # We use i + 1 so the files are numbered 1 through 10
        filename = f"{season}_{i + 1}.csv"
    
    # Indices 10-19 are Winter
    else:
        season = "Winter"
        # We subtract 9 so the numbering starts at 1 (10-9=1, 11-9=2, etc.)
        filename = f"{season}_{i - 9}.csv"
    
    # Save the dataframe to the folder
    file_path = os.path.join(output_folder, filename)
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

Saved: load_zone_results\Summer_1.csv
Saved: load_zone_results\Summer_2.csv
Saved: load_zone_results\Summer_3.csv
Saved: load_zone_results\Summer_4.csv
Saved: load_zone_results\Summer_5.csv
Saved: load_zone_results\Summer_6.csv
Saved: load_zone_results\Summer_7.csv
Saved: load_zone_results\Summer_8.csv
Saved: load_zone_results\Summer_9.csv
Saved: load_zone_results\Summer_10.csv
Saved: load_zone_results\Winter_1.csv
Saved: load_zone_results\Winter_2.csv
Saved: load_zone_results\Winter_3.csv
Saved: load_zone_results\Winter_4.csv
Saved: load_zone_results\Winter_5.csv
Saved: load_zone_results\Winter_6.csv
Saved: load_zone_results\Winter_7.csv
Saved: load_zone_results\Winter_8.csv
Saved: load_zone_results\Winter_9.csv
Saved: load_zone_results\Winter_10.csv


In [29]:
summers = [] 
winters = []  

winter_dfs = []
summer_dfs = []

for i in range(1, 11): 
    summers.append(f"load_zone_results/Summer_{i}.csv")
    winters.append(f"load_zone_results/Winter_{i}.csv")

for i in range(0, 10): 
    winter_dfs.append(pd.read_csv(winters[i]))
    summer_dfs.append(pd.read_csv(summers[i]))

for i in range(10): 
    # Ensure the source column exists before assignment
    if "thi" in dfs[i].columns:
        # We use .copy() to ensure we aren't modifying a view
        summer_dfs[i] = summer_dfs[i].copy()
        summer_dfs[i]["thi"] = dfs[i]["thi"].values # Use .values to ignore index mismatches
    
    winter_index = i + 10
    if "thi" in dfs[winter_index].columns:
        winter_dfs[i] = winter_dfs[i].copy()
        winter_dfs[i]["thi"] = dfs[winter_index]["thi"].values

In [30]:
print(type(summer_dfs))

<class 'list'>


In [31]:
import os

# Create a folder to store the results if it doesn't exist
output_folder = "load_zone_results"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
for i, df in enumerate(summer_dfs):
    filename = f"Summer_{i + 1}.csv"
    file_path = os.path.join(output_folder, filename)
    
    # df is the actual DataFrame object now
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

# Processing Winter DataFrames
for i, df in enumerate(winter_dfs):
    filename = f"Winter_{i + 1}.csv"
    file_path = os.path.join(output_folder, filename)
    
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

Saved: load_zone_results\Summer_1.csv
Saved: load_zone_results\Summer_2.csv
Saved: load_zone_results\Summer_3.csv
Saved: load_zone_results\Summer_4.csv
Saved: load_zone_results\Summer_5.csv
Saved: load_zone_results\Summer_6.csv
Saved: load_zone_results\Summer_7.csv
Saved: load_zone_results\Summer_8.csv
Saved: load_zone_results\Summer_9.csv
Saved: load_zone_results\Summer_10.csv
Saved: load_zone_results\Winter_1.csv
Saved: load_zone_results\Winter_2.csv
Saved: load_zone_results\Winter_3.csv
Saved: load_zone_results\Winter_4.csv
Saved: load_zone_results\Winter_5.csv
Saved: load_zone_results\Winter_6.csv
Saved: load_zone_results\Winter_7.csv
Saved: load_zone_results\Winter_8.csv
Saved: load_zone_results\Winter_9.csv
Saved: load_zone_results\Winter_10.csv
